In [ ]:
import pandas
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
train = pandas.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pandas.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
train.head()

# Categorical Missing Values

In [ ]:
target = train['SalePrice']
test_ids = test['Id']

train = train.drop(['SalePrice', 'Id'], axis = 1)
test = test.drop(['Id'], axis = 1)

dataset = pandas.concat([train, test], axis = 0)
dataset

In [ ]:
dataset.select_dtypes('object').loc[:, dataset.isna().sum()> 0].columns

In [ ]:
categorical_with_none = [
    'Alley',
    'MasVnrType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageType',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
    'MiscFeature',   
]
categorical = dataset.select_dtypes('object').loc[:, dataset.isna().sum() > 0].columns
categorical

In [ ]:
for col in categorical_with_none:
    dataset[col] = dataset[col].fillna('None')
for col in categorical:
    mod = dataset[col].mode()[0]
    dataset[col] = dataset[col].fillna(mod)

In [ ]:
dataset.select_dtypes('object').loc[:, dataset.isna().sum()> 0].columns

# Numeric Missing Values

In [ ]:
dataset.select_dtypes(np.number).loc[:, dataset.isna().sum() > 0].columns

In [ ]:
def FillWithKNN(dataset, ToFill):
    df = dataset.copy()

    numeric_df = df.select_dtypes(np.number)
    non_na_columns = numeric_df.loc[:, numeric_df.isna().sum() == 0].columns
    y_train = numeric_df.loc[numeric_df[ToFill].isna() == False, ToFill]
    X_train = numeric_df.loc[numeric_df[ToFill].isna() == False, non_na_columns]
    X_test = numeric_df.loc[numeric_df[ToFill].isna() == True, non_na_columns]
    
    knn = KNeighborsRegressor(n_neighbors = 10)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)

    df.loc[df[ToFill].isna() == True, ToFill] = y_pred
    return df
    

In [ ]:
na_columns = dataset.columns[(dataset.isna().sum()) > 0]
for column in na_columns:
    dataset = FillWithKNN(dataset, column)

In [ ]:
print(train.shape, test.shape, dataset.shape)
train = dataset.iloc[:train.shape[0], :].copy()
test = dataset.iloc[train.shape[0]:, :].copy()
full = [train, test]
print(train.shape, test.shape, dataset.shape)

In [ ]:
for ds in full:
    categories = ds.select_dtypes('object').columns
    print(len(categories))
    for column in categories:
        dic = pandas.concat([ds[column], target], axis = 1).groupby(column).mean().sort_values(by = 'SalePrice')
        mp = {}
        p = 0
        for it in dic.index:
            mp[it] = p
            p += 1
        ds[column] = ds[column].map(mp)
    categories = ds.select_dtypes('object').columns
    print(len(categories))

In [ ]:
X = train.copy()
y = target.copy()
X = StandardScaler().fit_transform(X)
test = StandardScaler().fit_transform(test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 8, shuffle = True)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
acc = {}

In [ ]:
DT_reg = RandomForestRegressor(n_estimators = 100)
DT_reg.fit(X_train, y_train)
pred = DT_reg.predict(X_test)
DT_TestAccuracy = mean_squared_error(pred, y_test)
print("Decision Tree Testing Accuracy: ", round(DT_TestAccuracy, 2))
acc['DecisionTree'] = DT_TestAccuracy

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
pred = lin_reg.predict(X_test)
lin_TestAccuracy = mean_squared_error(pred, y_test)
print("Linear Regression Testing Accuracy: ", round(lin_TestAccuracy, 2))
acc['Linear Regression'] = lin_TestAccuracy

In [ ]:
ridge_reg = Ridge(alpha = 0.01)
ridge_reg.fit(X_train, y_train)
pred = ridge_reg.predict(X_test)
Ridge_TestAccuracy = mean_squared_error(pred, y_test)
print("Ridge Regression Testing Accuracy: ", round(Ridge_TestAccuracy, 2))
acc['Ridge'] = Ridge_TestAccuracy

In [ ]:
y_pred = DT_reg.predict(test)

In [ ]:
submission = pandas.DataFrame({
        "Id": range(1461, 2920),
        "SalePrice": y_pred
    })

In [ ]:
submission.to_csv('submission.csv', index = False)

In [ ]:
submission